In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
!nvidia-smi

Tue Jan  6 00:53:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

# 1. Load Environment

In [5]:
!pip install -q transformers datasets accelerate peft bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 39.0 MB/s eta 0:00:00


In [6]:
import transformers
import datasets
import accelerate
import peft
import bitsandbytes
import trl

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("trl:", trl.__version__)

transformers: 4.57.3
datasets: 4.0.0
accelerate: 1.12.0
peft: 0.18.0
bitsandbytes: 0.49.0
trl: 0.26.2


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 2. Import Libraries

In [8]:
import math
import numpy as np
from datasets import Dataset, load_dataset, load_from_disk
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score

# 3. Load Dataset

In [9]:
name_dataset = 'DB_Pedia'

In [10]:
dataset = load_dataset("dbpedia_14")

README.md: 0.00B [00:00, ?B/s]

dbpedia_14/train-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

dbpedia_14/test-00000-of-00001.parquet:   0%|          | 0.00/13.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/560000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/70000 [00:00<?, ? examples/s]

In [11]:
dataset = dataset.remove_columns(["title"])

In [12]:
dataset = dataset.rename_column("content", "text")

In [13]:
dataset.shape

{'train': (560000, 2), 'test': (70000, 2)}

In [14]:
dataset_train = dataset['train']
pre_dataset_test = dataset['test']

In [15]:
split_test = pre_dataset_test.train_test_split(test_size=0.1, stratify_by_column = 'label', seed=42)

In [16]:
dataset_val = split_test['train']
dataset_test = split_test['test']

In [17]:
df_train = dataset_train.to_pandas()
df_val = dataset_val.to_pandas()
df_test = dataset_test.to_pandas()

**a. Analysis: Train Set**

In [18]:
df_train.shape

(560000, 2)

In [19]:
df_train['label'].value_counts()

,count
label,
0,40000
1,40000
2,40000
3,40000
4,40000
5,40000
6,40000
7,40000
8,40000


In [20]:
round(df_train['label'].value_counts(normalize = True)*100, 2)

,proportion
label,
0,7.14
1,7.14
2,7.14
3,7.14
4,7.14
5,7.14
6,7.14
7,7.14
8,7.14


**b. Analysis: Validation Set**

In [21]:
df_val.shape

(63000, 2)

In [22]:
df_val['label'].value_counts()

,count
label,
11,4500
6,4500
10,4500
1,4500
0,4500
4,4500
7,4500
8,4500
3,4500


In [23]:
round(df_val['label'].value_counts(normalize = True)*100, 2)

,proportion
label,
11,7.14
6,7.14
10,7.14
1,7.14
0,7.14
4,7.14
7,7.14
8,7.14
3,7.14


**c. Analysis: Test Set**

In [24]:
df_test.shape

(7000, 2)

In [25]:
df_test['label'].value_counts()

,count
label,
5,500
3,500
9,500
2,500
1,500
10,500
8,500
6,500
12,500


In [26]:
round(df_test['label'].value_counts(normalize = True)*100, 2)

,proportion
label,
5,7.14
3,7.14
9,7.14
2,7.14
1,7.14
10,7.14
8,7.14
6,7.14
12,7.14


**d. Save dataframes**

In [27]:
path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/01.Datasets_Creation/{name_dataset}'

In [28]:
df_train.to_csv(f'{path_save}/df_train.csv')
df_val.to_csv(f'{path_save}/df_val.csv')
df_test.to_csv(f'{path_save}/df_test.csv')

In [29]:
N = 5
chunk_size = math.ceil(len(df_test) / N)
splits_test = [
    df_test.iloc[i:i + chunk_size]
    for i in range(0, len(df_test), chunk_size)
]

In [30]:
print(len(splits_test))
print(splits_test[0].shape)

5
(1400, 2)


In [31]:
for i in range(N):
  idx = i + 1
  splits_test[i].to_csv(f'{path_save}/df_test_{idx}.csv')

# 4. BERT

In [32]:
name_model = "bert-base-uncased"

In [33]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [34]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [35]:
tokenized_train = dataset_train.map(preprocess_function, batched = True)

Map:   0%|          | 0/560000 [00:00<?, ? examples/s]

In [36]:
tokenized_val = dataset_val.map(preprocess_function, batched = True)

Map:   0%|          | 0/63000 [00:00<?, ? examples/s]

In [37]:
tokenized_test = dataset_test.map(preprocess_function, batched = True)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [38]:
tokenized_train.save_to_disk(f'{path_save}/{name_model}/train')

Saving the dataset (0/1 shards):   0%|          | 0/560000 [00:00<?, ? examples/s]

In [39]:
tokenized_val.save_to_disk(f'{path_save}/{name_model}/val')

Saving the dataset (0/1 shards):   0%|          | 0/63000 [00:00<?, ? examples/s]

In [40]:
tokenized_test.save_to_disk(f'{path_save}/{name_model}/test')

Saving the dataset (0/1 shards):   0%|          | 0/7000 [00:00<?, ? examples/s]

# 5. DistilBERT

In [41]:
name_model = "distilbert-base-uncased"

In [42]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [43]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [44]:
tokenized_train = dataset_train.map(preprocess_function, batched = True)

Map:   0%|          | 0/560000 [00:00<?, ? examples/s]

In [45]:
tokenized_val = dataset_val.map(preprocess_function, batched = True)

Map:   0%|          | 0/63000 [00:00<?, ? examples/s]

In [46]:
tokenized_test = dataset_test.map(preprocess_function, batched = True)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [47]:
tokenized_train.save_to_disk(f'{path_save}/{name_model}/train')

Saving the dataset (0/1 shards):   0%|          | 0/560000 [00:00<?, ? examples/s]

In [48]:
tokenized_val.save_to_disk(f'{path_save}/{name_model}/val')

Saving the dataset (0/1 shards):   0%|          | 0/63000 [00:00<?, ? examples/s]

In [49]:
tokenized_test.save_to_disk(f'{path_save}/{name_model}/test')

Saving the dataset (0/1 shards):   0%|          | 0/7000 [00:00<?, ? examples/s]

# 6. RoBERTa

In [50]:
name_model = "roberta-base"

In [51]:
tokenizer = AutoTokenizer.from_pretrained(name_model)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [52]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation = True)

In [53]:
tokenized_train = dataset_train.map(preprocess_function, batched = True)

Map:   0%|          | 0/560000 [00:00<?, ? examples/s]

In [54]:
tokenized_val = dataset_val.map(preprocess_function, batched = True)

Map:   0%|          | 0/63000 [00:00<?, ? examples/s]

In [55]:
tokenized_test = dataset_test.map(preprocess_function, batched = True)

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

In [56]:
tokenized_train.save_to_disk(f'{path_save}/{name_model}/train')

Saving the dataset (0/1 shards):   0%|          | 0/560000 [00:00<?, ? examples/s]

In [57]:
tokenized_val.save_to_disk(f'{path_save}/{name_model}/val')

Saving the dataset (0/1 shards):   0%|          | 0/63000 [00:00<?, ? examples/s]

In [58]:
tokenized_test.save_to_disk(f'{path_save}/{name_model}/test')

Saving the dataset (0/1 shards):   0%|          | 0/7000 [00:00<?, ? examples/s]

# 7. Execution Time

In [59]:
end_notebook = time.time()

In [60]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 0h 5m 1.93s
